# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [2]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [7]:
# Part 1 - Question 1
# Add a unique key to identify each trip.
# Also prepare the columns needed for the date and time questions.

from pyspark.sql import functions as F

df_part1 = (
    df_trips
    .withColumn(
        "trip_id",
        F.monotonically_increasing_id(),
    )
    .withColumn(
        "trip_duration_minutes",
        F.timestamp_diff(
            "SECOND",
            "tpep_pickup_datetime",
            "tpep_dropoff_datetime",
        ) / 60,
    )
    .withColumn(
        "pickup_date",
        F.to_date("tpep_pickup_datetime"),
    )
    .withColumn(
        "pickup_hour",
        F.hour("tpep_pickup_datetime"),
    )
    .withColumn(
        "pickup_day",
        F.date_format("tpep_pickup_datetime", "EEEE"),
    )
)

# Keep only trips that started in January 2019.
df_jan = df_part1.filter(
    (F.col("tpep_pickup_datetime") >= "2019-01-01")
    & (F.col("tpep_pickup_datetime") < "2019-02-01")
)

df_jan.select(
    "trip_id",
    "tpep_pickup_datetime",
    "trip_duration_minutes",
    "pickup_date",
    "pickup_hour",
    "pickup_day",
).show(5, truncate=False)

+-----------+--------------------+---------------------+-----------+-----------+----------+
|trip_id    |tpep_pickup_datetime|trip_duration_minutes|pickup_date|pickup_hour|pickup_day|
+-----------+--------------------+---------------------+-----------+-----------+----------+
|42949672960|2019-01-01 00:46:40 |6.666666666666667    |2019-01-01 |0          |Tuesday   |
|42949672961|2019-01-01 00:59:47 |19.2                 |2019-01-01 |0          |Tuesday   |
|42949672967|2019-01-01 00:21:28 |7.15                 |2019-01-01 |0          |Tuesday   |
|42949672968|2019-01-01 00:32:01 |13.633333333333333   |2019-01-01 |0          |Tuesday   |
|42949672969|2019-01-01 00:57:32 |12.0                 |2019-01-01 |0          |Tuesday   |
+-----------+--------------------+---------------------+-----------+-----------+----------+
only showing top 5 rows


In [8]:
# Part 1 - Questions 2 and 3
# Which trip has the highest passenger count?
# What is the average passenger count?

print("Trip with the highest passenger count:")

df_jan.orderBy(
    F.col("passenger_count").desc_nulls_last()
).select(
    "trip_id",
    "tpep_pickup_datetime",
    "passenger_count",
    "trip_distance",
).show(1, truncate=False)

print("Average passenger count:")

df_jan.select(
    F.avg("passenger_count").alias(
        "average_passenger_count"
    )
).show()

Trip with the highest passenger count:
+-----------+--------------------+---------------+-------------+
|trip_id    |tpep_pickup_datetime|passenger_count|trip_distance|
+-----------+--------------------+---------------+-------------+
|42950622916|2019-01-05 13:12:29 |9.0            |0.0          |
+-----------+--------------------+---------------+-------------+
only showing top 1 row
Average passenger count:
+-----------------------+
|average_passenger_count|
+-----------------------+
|     1.5670089553079738|
+-----------------------+



In [9]:
# Part 1 - Question 4
# What are the shortest and longest trips by distance and time?

# Keep only positive distances.
valid_distance = df_jan.filter(
    F.col("trip_distance") > 0
)

print("Shortest positive trip by distance:")

valid_distance.orderBy(
    F.col("trip_distance").asc()
).select(
    "trip_id",
    "trip_distance",
    "trip_duration_minutes",
).show(1, truncate=False)

print("Longest trip by distance:")

valid_distance.orderBy(
    F.col("trip_distance").desc()
).select(
    "trip_id",
    "trip_distance",
    "trip_duration_minutes",
).show(1, truncate=False)

# Keep only positive durations.
valid_duration = df_jan.filter(
    F.col("trip_duration_minutes") > 0
)

print("Shortest positive trip by time:")

valid_duration.orderBy(
    F.col("trip_duration_minutes").asc()
).select(
    "trip_id",
    "trip_distance",
    "trip_duration_minutes",
).show(1, truncate=False)

print("Longest trip by time:")

valid_duration.orderBy(
    F.col("trip_duration_minutes").desc()
).select(
    "trip_id",
    "trip_distance",
    "trip_duration_minutes",
).show(1, truncate=False)

Shortest positive trip by distance:
+-----------+-------------+---------------------+
|trip_id    |trip_distance|trip_duration_minutes|
+-----------+-------------+---------------------+
|42949685524|0.01         |0.06666666666666667  |
+-----------+-------------+---------------------+
only showing top 1 row
Longest trip by distance:
+-----------+-------------+---------------------+
|trip_id    |trip_distance|trip_duration_minutes|
+-----------+-------------+---------------------+
|42955747051|831.8        |9.483333333333333    |
+-----------+-------------+---------------------+
only showing top 1 row
Shortest positive trip by time:
+-----------+-------------+---------------------+
|trip_id    |trip_distance|trip_duration_minutes|
+-----------+-------------+---------------------+
|42949685315|0.0          |0.016666666666666666 |
+-----------+-------------+---------------------+
only showing top 1 row
Longest trip by time:
+-----------+-------------+---------------------+
|trip_id    |tr

In [10]:
# Part 1 - Question 5
# What are the busiest and slowest individual days?

daily_trips = (
    df_jan
    .groupBy("pickup_date")
    .count()
)

print("Busiest day:")

daily_trips.orderBy(
    F.col("count").desc()
).show(1)

print("Slowest day:")

daily_trips.orderBy(
    F.col("count").asc()
).show(1)

Busiest day:
+-----------+------+
|pickup_date| count|
+-----------+------+
| 2019-01-25|292499|
+-----------+------+
only showing top 1 row
Slowest day:
+-----------+------+
|pickup_date| count|
+-----------+------+
| 2019-01-01|189432|
+-----------+------+
only showing top 1 row


In [11]:
# Part 1 - Question 6
# What are the busiest and slowest hours of the day?

hourly_trips = (
    df_jan
    .groupBy("pickup_hour")
    .count()
)

print("Busiest hour:")

hourly_trips.orderBy(
    F.col("count").desc()
).show(1)

print("Slowest hour:")

hourly_trips.orderBy(
    F.col("count").asc()
).show(1)

Busiest hour:
+-----------+------+
|pickup_hour| count|
+-----------+------+
|         18|515374|
+-----------+------+
only showing top 1 row
Slowest hour:
+-----------+-----+
|pickup_hour|count|
+-----------+-----+
|          4|61423|
+-----------+-----+
only showing top 1 row


In [12]:
# Part 1 - Question 7
# On average, which weekday is the busiest or slowest?

# Count trips for every individual date.
daily_counts = (
    df_jan
    .groupBy(
        "pickup_date",
        "pickup_day",
    )
    .count()
)

# Calculate the average for each weekday.
average_by_weekday = (
    daily_counts
    .groupBy("pickup_day")
    .agg(
        F.avg("count").alias(
            "average_daily_trips"
        )
    )
)

average_by_weekday.orderBy(
    F.col("average_daily_trips").desc()
).show()

+----------+-------------------+
|pickup_day|average_daily_trips|
+----------+-------------------+
|    Friday|           271787.5|
|  Thursday|           271398.4|
| Wednesday|           253045.8|
|  Saturday|          252494.75|
|   Tuesday|           241815.2|
|    Monday|           226941.0|
|    Sunday|           214972.5|
+----------+-------------------+



In [13]:
# Part 1 - Question 8
# Do trip distance or passenger count affect the tip amount?

# Remove clearly invalid values.
valid_tip_data = df_jan.filter(
    (F.col("trip_distance") >= 0)
    & (F.col("trip_distance") <= 100)
    & (F.col("passenger_count") >= 1)
    & (F.col("passenger_count") <= 8)
    & (F.col("tip_amount") >= 0)
)

print("Correlations with tip amount:")

valid_tip_data.select(
    F.corr(
        "trip_distance",
        "tip_amount",
    ).alias("distance_tip_correlation"),
    F.corr(
        "passenger_count",
        "tip_amount",
    ).alias("passenger_tip_correlation"),
).show()

print("Average tip by passenger count:")

(
    valid_tip_data
    .groupBy("passenger_count")
    .agg(
        F.avg("tip_amount").alias("average_tip"),
        F.count("*").alias("number_of_trips"),
    )
    .orderBy("passenger_count")
    .show()
)

Correlations with tip amount:
+------------------------+-------------------------+
|distance_tip_correlation|passenger_tip_correlation|
+------------------------+-------------------------+
|      0.5434331799642338|     7.619041951528298E-4|
+------------------------+-------------------------+

Average tip by passenger count:
+---------------+------------------+---------------+
|passenger_count|       average_tip|number_of_trips|
+---------------+------------------+---------------+
|            1.0| 1.828435501426751|        5456074|
|            2.0|1.8340068344337352|        1113772|
|            3.0| 1.795489236479714|         314674|
|            4.0|1.7024177034410155|         140741|
|            5.0|1.8699804191656964|         323786|
|            6.0|1.8568146683534568|         200786|
|            7.0| 6.542631578947368|             19|
|            8.0| 6.480689655172414|             29|
+---------------+------------------+---------------+



In [15]:
# Part 1 - Question 9
# What was the highest extra charge and which trip had it?

print("Trip with the highest extra charge:")

df_jan.orderBy(
    F.col("extra").desc_nulls_last()
).select(
    "trip_id",
    "tpep_pickup_datetime",
    "trip_distance",
    "passenger_count",
    "fare_amount",
    "extra",
    "tip_amount",
    "total_amount",
).show(1, truncate=False)

Trip with the highest extra charge:
+-----------+--------------------+-------------+---------------+-----------+------+----------+------------+
|trip_id    |tpep_pickup_datetime|trip_distance|passenger_count|fare_amount|extra |tip_amount|total_amount|
+-----------+--------------------+-------------+---------------+-----------+------+----------+------------+
|42954996443|2019-01-23 08:58:09 |0.0          |1.0            |355676.98  |535.38|0.0       |356214.78   |
+-----------+--------------------+-------------+---------------+-----------+------+----------+------------+
only showing top 1 row


In [16]:
# Part 1 - Question 10
# Are there strange data points or possible outliers?

print("Minimum and maximum values:")

df_part1.select(
    F.min("tpep_pickup_datetime").alias("first_date"),
    F.max("tpep_pickup_datetime").alias("last_date"),
    F.min("passenger_count").alias("minimum_passengers"),
    F.max("passenger_count").alias("maximum_passengers"),
    F.min("trip_distance").alias("minimum_distance"),
    F.max("trip_distance").alias("maximum_distance"),
    F.min("trip_duration_minutes").alias(
        "minimum_duration"
    ),
    F.max("trip_duration_minutes").alias(
        "maximum_duration"
    ),
    F.min("fare_amount").alias("minimum_fare"),
    F.max("fare_amount").alias("maximum_fare"),
    F.max("extra").alias("maximum_extra"),
).show(truncate=False)

print("Examples of suspicious trips:")

df_part1.filter(
    (F.col("trip_distance") < 0)
    | (F.col("trip_distance") > 500)
    | (F.col("trip_duration_minutes") <= 0)
    | (F.col("trip_duration_minutes") > 1440)
    | (F.col("passenger_count") > 8)
    | (F.col("fare_amount") < 0)
    | (F.col("fare_amount") > 1000)
    | (F.col("tpep_pickup_datetime") < "2019-01-01")
    | (F.col("tpep_pickup_datetime") >= "2019-02-01")
).select(
    "trip_id",
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "trip_duration_minutes",
    "fare_amount",
    "extra",
    "total_amount",
).show(20, truncate=False)

Minimum and maximum values:
+-------------------+-------------------+------------------+------------------+----------------+----------------+----------------+-----------------+------------+------------+-------------+
|first_date         |last_date          |minimum_passengers|maximum_passengers|minimum_distance|maximum_distance|minimum_duration|maximum_duration |minimum_fare|maximum_fare|maximum_extra|
+-------------------+-------------------+------------------+------------------+----------------+----------------+----------------+-----------------+------------+------------+-------------+
|2001-02-02 14:55:07|2088-01-24 00:25:39|0.0               |9.0               |0.0             |831.8           |-84280.5        |43648.01666666667|-362.0      |623259.86   |535.38       |
+-------------------+-------------------+------------------+------------------+----------------+----------------+----------------+-----------------+------------+------------+-------------+

Examples of suspicious tri

##### Part 1 - Question 10: Outlier explanation

I found several values that do not look normal. The file is supposed
to contain trips from January 2019, but some dates are between 2001
and 2088.

The longest distance is 831.8 miles and some durations are negative.
One trip lasts more than 43,000 minutes, which is about 30 days. These
values are not realistic for a taxi trip.

There are also negative fares and one fare is over 600,000 dollars.
The highest extra charge is more than 500 dollars. These values may
come from data entry errors, GPS problems, cancelled trips or payment
corrections.

For this reason, invalid dates, negative values and very large values
should be filtered before using the data for a real analysis.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [17]:
# Part 2 - Question 1
# Load and explore the taxi zone lookup table.

zone_url = (
    "https://d37ci6vzurychx.cloudfront.net/"
    "misc/taxi_zone_lookup.csv"
)

zone_file = "taxi_zone_lookup.csv"

response = requests.get(zone_url)
response.raise_for_status()

with open(zone_file, "wb") as file:
    file.write(response.content)

df_zones = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(zone_file)
)

print("Number of taxi zones:", df_zones.count())

df_zones.show(10, truncate=False)

df_zones.printSchema()

Number of taxi zones: 265
+----------+-------------+-----------------------+------------+
|LocationID|Borough      |Zone                   |service_zone|
+----------+-------------+-----------------------+------------+
|1         |EWR          |Newark Airport         |EWR         |
|2         |Queens       |Jamaica Bay            |Boro Zone   |
|3         |Bronx        |Allerton/Pelham Gardens|Boro Zone   |
|4         |Manhattan    |Alphabet City          |Yellow Zone |
|5         |Staten Island|Arden Heights          |Boro Zone   |
|6         |Staten Island|Arrochar/Fort Wadsworth|Boro Zone   |
|7         |Queens       |Astoria                |Boro Zone   |
|8         |Queens       |Astoria Park           |Boro Zone   |
|9         |Queens       |Auburndale             |Boro Zone   |
|10        |Queens       |Baisley Park           |Boro Zone   |
+----------+-------------+-----------------------+------------+
only showing top 10 rows
root
 |-- LocationID: integer (nullable = true)
 |-- 

In [18]:
# Part 2 - Question 2
# Which borough had the most pickups and dropoffs?

# Prepare the zone table for pickup locations.
pickup_zones = df_zones.select(
    F.col("LocationID").alias("PULocationID"),
    F.col("Borough").alias("pickup_borough"),
    F.col("Zone").alias("pickup_zone"),
)

# Prepare the zone table for dropoff locations.
dropoff_zones = df_zones.select(
    F.col("LocationID").alias("DOLocationID"),
    F.col("Borough").alias("dropoff_borough"),
    F.col("Zone").alias("dropoff_zone"),
)

# Join the trip data with both zone tables.
df_trips_zones = (
    df_jan
    .join(
        pickup_zones,
        on="PULocationID",
        how="left",
    )
    .join(
        dropoff_zones,
        on="DOLocationID",
        how="left",
    )
)

print("Pickups by borough:")

(
    df_trips_zones
    .groupBy("pickup_borough")
    .count()
    .orderBy(F.col("count").desc())
    .show(truncate=False)
)

print("Dropoffs by borough:")

(
    df_trips_zones
    .groupBy("dropoff_borough")
    .count()
    .orderBy(F.col("count").desc())
    .show(truncate=False)
)

Pickups by borough:
+--------------+-------+
|pickup_borough|count  |
+--------------+-------+
|Manhattan     |6950511|
|Queens        |471113 |
|Unknown       |159807 |
|Brooklyn      |91896  |
|Bronx         |18056  |
|N/A           |3890   |
|EWR           |446    |
|Staten Island |361    |
+--------------+-------+

Dropoffs by borough:
+---------------+-------+
|dropoff_borough|count  |
+---------------+-------+
|Manhattan      |6816936|
|Queens         |340914 |
|Brooklyn       |301074 |
|Unknown        |149091 |
|Bronx          |58068  |
|N/A            |16900  |
|EWR            |10913  |
|Staten Island  |2184   |
+---------------+-------+



In [19]:
# Part 2 - Question 3
# Find the busiest and slowest pickup hour for each borough.

from pyspark.sql.window import Window

boroughs = [
    "Manhattan",
    "Queens",
    "Brooklyn",
    "Bronx",
    "Staten Island",
]

borough_hour_counts = (
    df_trips_zones
    .filter(F.col("pickup_borough").isin(boroughs))
    .groupBy(
        "pickup_borough",
        "pickup_hour",
    )
    .count()
)

# Rank hours from busiest to slowest inside each borough.
busy_window = (
    Window
    .partitionBy("pickup_borough")
    .orderBy(
        F.col("count").desc(),
        F.col("pickup_hour").asc(),
    )
)

busiest_hours = (
    borough_hour_counts
    .withColumn(
        "rank",
        F.row_number().over(busy_window),
    )
    .filter(F.col("rank") == 1)
    .select(
        "pickup_borough",
        F.col("pickup_hour").alias("busiest_hour"),
        F.col("count").alias("busiest_trip_count"),
    )
)

# Rank hours from slowest to busiest inside each borough.
slow_window = (
    Window
    .partitionBy("pickup_borough")
    .orderBy(
        F.col("count").asc(),
        F.col("pickup_hour").asc(),
    )
)

slowest_hours = (
    borough_hour_counts
    .withColumn(
        "rank",
        F.row_number().over(slow_window),
    )
    .filter(F.col("rank") == 1)
    .select(
        "pickup_borough",
        F.col("pickup_hour").alias("slowest_hour"),
        F.col("count").alias("slowest_trip_count"),
    )
)

# Combine the busiest and slowest results.
(
    busiest_hours
    .join(
        slowest_hours,
        on="pickup_borough",
    )
    .orderBy("pickup_borough")
    .show(truncate=False)
)

+--------------+------------+------------------+------------+------------------+
|pickup_borough|busiest_hour|busiest_trip_count|slowest_hour|slowest_trip_count|
+--------------+------------+------------------+------------+------------------+
|Bronx         |7           |1803              |2           |225               |
|Brooklyn      |8           |6935              |3           |1919              |
|Manhattan     |18          |471524            |4           |53446             |
|Queens        |16          |29880             |3           |3084              |
|Staten Island |8           |36                |1           |3                 |
+--------------+------------+------------------+------------+------------------+



In [20]:
# Part 2 - Question 4
# Find the busiest weekday on average for each borough.

# Count trips for each date and borough.
borough_daily_counts = (
    df_trips_zones
    .filter(F.col("pickup_borough").isin(boroughs))
    .groupBy(
        "pickup_borough",
        "pickup_date",
        "pickup_day",
    )
    .count()
)

# Calculate the daily average for each weekday and borough.
borough_weekday_averages = (
    borough_daily_counts
    .groupBy(
        "pickup_borough",
        "pickup_day",
    )
    .agg(
        F.avg("count").alias("average_trip_count")
    )
)

# Rank weekdays inside each borough.
weekday_window = (
    Window
    .partitionBy("pickup_borough")
    .orderBy(
        F.col("average_trip_count").desc()
    )
)

busiest_weekdays = (
    borough_weekday_averages
    .withColumn(
        "rank",
        F.row_number().over(weekday_window),
    )
    .filter(F.col("rank") == 1)
    .drop("rank")
    .orderBy("pickup_borough")
)

busiest_weekdays.show(truncate=False)

+--------------+----------+------------------+
|pickup_borough|pickup_day|average_trip_count|
+--------------+----------+------------------+
|Bronx         |Friday    |666.5             |
|Brooklyn      |Friday    |3273.0            |
|Manhattan     |Friday    |246224.0          |
|Queens        |Monday    |16662.0           |
|Staten Island |Friday    |16.0              |
+--------------+----------+------------------+



In [21]:
# Part 2 - Question 5
# Calculate the average trip distance by pickup borough.

average_distance_by_borough = (
    df_trips_zones
    .filter(
        F.col("pickup_borough").isin(boroughs)
        & (F.col("trip_distance") >= 0)
        & (F.col("trip_distance") <= 100)
    )
    .groupBy("pickup_borough")
    .agg(
        F.avg("trip_distance").alias(
            "average_trip_distance"
        ),
        F.count("*").alias("number_of_trips"),
    )
    .orderBy(
        F.col("average_trip_distance").desc()
    )
)

average_distance_by_borough.show(truncate=False)

+--------------+---------------------+---------------+
|pickup_borough|average_trip_distance|number_of_trips|
+--------------+---------------------+---------------+
|Staten Island |12.503601108033246   |361            |
|Queens        |11.280068032553483   |471098         |
|Bronx         |7.223364165051208    |18055          |
|Brooklyn      |4.786131889656692    |91895          |
|Manhattan     |2.228265895978548    |6950500        |
+--------------+---------------------+---------------+



In [22]:
# Part 2 - Question 6
# Calculate the average fare amount by pickup borough.

average_fare_by_borough = (
    df_trips_zones
    .filter(
        F.col("pickup_borough").isin(boroughs)
        & (F.col("fare_amount") >= 0)
        & (F.col("fare_amount") <= 1000)
    )
    .groupBy("pickup_borough")
    .agg(
        F.avg("fare_amount").alias(
            "average_fare_amount"
        ),
        F.count("*").alias("number_of_trips"),
    )
    .orderBy(
        F.col("average_fare_amount").desc()
    )
)

average_fare_by_borough.show(truncate=False)

+--------------+-------------------+---------------+
|pickup_borough|average_fare_amount|number_of_trips|
+--------------+-------------------+---------------+
|Staten Island |46.14408450704227  |355            |
|Queens        |35.25100457815466  |470058         |
|Bronx         |26.52268866347212  |17942          |
|Brooklyn      |18.73894390880223  |91583          |
|Manhattan     |10.665337822536992 |6945259        |
+--------------+-------------------+---------------+



In [23]:
# Part 2 - Question 7
# Find the trips with the highest and lowest fare amounts.
# Show the pickup borough associated with each trip.

known_borough_trips = df_trips_zones.filter(
    F.col("pickup_borough").isin(boroughs)
)

print("Trip with the highest fare amount:")

known_borough_trips.orderBy(
    F.col("fare_amount").desc_nulls_last()
).select(
    "trip_id",
    "tpep_pickup_datetime",
    "pickup_borough",
    "pickup_zone",
    "dropoff_borough",
    "dropoff_zone",
    "trip_distance",
    "fare_amount",
    "total_amount",
).show(1, truncate=False)

print("Trip with the lowest fare amount:")

known_borough_trips.orderBy(
    F.col("fare_amount").asc_nulls_last()
).select(
    "trip_id",
    "tpep_pickup_datetime",
    "pickup_borough",
    "pickup_zone",
    "dropoff_borough",
    "dropoff_zone",
    "trip_distance",
    "fare_amount",
    "total_amount",
).show(1, truncate=False)

Trip with the highest fare amount:
+-----------+--------------------+--------------+---------------------+---------------+------------+-------------+-----------+------------+
|trip_id    |tpep_pickup_datetime|pickup_borough|pickup_zone          |dropoff_borough|dropoff_zone|trip_distance|fare_amount|total_amount|
+-----------+--------------------+--------------+---------------------+---------------+------------+-------------+-----------+------------+
|42952172615|2019-01-11 19:33:15 |Manhattan     |Upper East Side South|Manhattan      |Flatiron    |2.4          |623259.86  |623261.66   |
+-----------+--------------------+--------------+---------------------+---------------+------------+-------------+-----------+------------+
only showing top 1 row
Trip with the lowest fare amount:
+-----------+--------------------+--------------+-----------+---------------+------------+-------------+-----------+------------+
|trip_id    |tpep_pickup_datetime|pickup_borough|pickup_zone|dropoff_borough|d

In [24]:
# Part 2 - Question 8
# Load the most recently available January dataset.

import os

recent_url = (
    "https://d37ci6vzurychx.cloudfront.net/"
    "trip-data/yellow_tripdata_2026-01.parquet"
)

recent_file = "yellow_tripdata_2026-01.parquet"

# Download only if the file is not already present.
if not os.path.exists(recent_file):
    with requests.get(recent_url, stream=True) as response:
        response.raise_for_status()

        with open(recent_file, "wb") as file:
            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):
                file.write(chunk)

    print("Download completed.")
else:
    print("File already downloaded.")

# Load the Parquet file.
df_2026 = spark.read.parquet(recent_file)

print("Number of trips in January 2026:")
print(df_2026.count())

df_2026.show(5, truncate=False)

Download completed.
Number of trips in January 2026:
3724889
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|2       |2026-01-01 00:54:04 |2026-01-01 00:59:37  |1              |

In [25]:
# Part 2 - Question 8
# Compare average metrics between January 2019 and January 2026.

def calculate_average_metrics(
    dataframe,
    year,
    start_date,
    end_date,
):
    # Apply the same filters to both datasets.
    valid_data = dataframe.filter(
        (F.col("tpep_pickup_datetime") >= start_date)
        & (F.col("tpep_pickup_datetime") < end_date)
        & (F.col("trip_distance") >= 0)
        & (F.col("trip_distance") <= 100)
        & (F.col("passenger_count") >= 0)
        & (F.col("passenger_count") <= 8)
        & (F.col("fare_amount") >= 0)
        & (F.col("fare_amount") <= 1000)
        & (F.col("tip_amount") >= 0)
    )

    return (
        valid_data
        .agg(
            F.count("*").alias("number_of_valid_trips"),
            F.avg("passenger_count").alias(
                "average_passengers"
            ),
            F.avg("trip_distance").alias(
                "average_distance"
            ),
            F.avg("fare_amount").alias(
                "average_fare"
            ),
            F.avg("tip_amount").alias(
                "average_tip"
            ),
            F.avg("total_amount").alias(
                "average_total"
            ),
        )
        .withColumn("year", F.lit(year))
        .select(
            "year",
            "number_of_valid_trips",
            "average_passengers",
            "average_distance",
            "average_fare",
            "average_tip",
            "average_total",
        )
    )


metrics_2019 = calculate_average_metrics(
    df_trips,
    "2019",
    "2019-01-01",
    "2019-02-01",
)

metrics_2026 = calculate_average_metrics(
    df_2026,
    "2026",
    "2026-01-01",
    "2026-02-01",
)

comparison = metrics_2019.unionByName(metrics_2026)

comparison.show(truncate=False)

# Calculate percentage changes.
results = {
    row["year"]: row.asDict()
    for row in comparison.collect()
}

metrics = [
    "number_of_valid_trips",
    "average_passengers",
    "average_distance",
    "average_fare",
    "average_tip",
    "average_total",
]

print("Percentage changes from 2019 to 2026:")

for metric in metrics:
    old_value = results["2019"][metric]
    new_value = results["2026"][metric]

    percentage_change = (
        (new_value - old_value) / old_value
    ) * 100

    print(
        f"{metric}: "
        f"{percentage_change:.2f}%"
    )

+----+---------------------+------------------+------------------+------------------+------------------+------------------+
|year|number_of_valid_trips|average_passengers|average_distance  |average_fare      |average_tip       |average_total     |
+----+---------------------+------------------+------------------+------------------+------------------+------------------+
|2019|7660219              |1.566955200628076 |2.8021392756526877|12.246825915812536|1.8291444317768966|15.523790724929148|
|2026|2597353              |1.2550527402320748|3.3796319907227828|19.56849811327239 |3.5834330258532714|29.03034549789499 |
+----+---------------------+------------------+------------------+------------------+------------------+------------------+

Percentage changes from 2019 to 2026:
number_of_valid_trips: -66.09%
average_passengers: -19.91%
average_distance: 20.61%
average_fare: 59.78%
average_tip: 95.91%
average_total: 87.01%


## Part 2 - Conclusion

Manhattan has the most pickups and dropoffs. Its busiest pickup hour
is 6 PM, while the other boroughs are generally busier during the
morning or afternoon. Friday is the busiest weekday for most boroughs,
but Monday is the busiest day in Queens.

Staten Island and Queens have the longest average trips and the
highest average fares. However, there are very few yellow taxi trips
starting in Staten Island, so its averages may not be very reliable.

The highest and lowest fares are clearly unusual. One fare is over
600,000 dollars and another fare is negative.

There are about 66% fewer valid trips in January 2026 than in January
2019. The average number of passengers also decreased. However, the
average distance, fare, tip and total amount increased. The average
total increased by about 87%.

These changes may be related to different travel habits, higher taxi
prices, inflation and new congestion charges. The results only show a
difference between the datasets and do not prove the exact cause.

### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

In [26]:
# Part 3 - SQL setup
# Make the DataFrames available as SQL tables.

df_jan.createOrReplaceTempView("trips_2019")
df_zones.createOrReplaceTempView("taxi_zones")

print("SQL tables created.")

SQL tables created.


In [28]:
# Part 3 - SQL Question 1
# Calculate the average passenger count using SQL.

average_passengers_sql = spark.sql("""
    SELECT
        ROUND(
            AVG(passenger_count),
            3
        ) AS average_passenger_count
    FROM trips_2019
""")

average_passengers_sql.show()

+-----------------------+
|average_passenger_count|
+-----------------------+
|                  1.567|
+-----------------------+



In [29]:
# Part 3 - SQL Question 2
# Find the busiest and slowest hours using SQL.

hours_sql = spark.sql("""
    WITH hourly_counts AS (
        SELECT
            pickup_hour,
            COUNT(*) AS trip_count
        FROM trips_2019
        GROUP BY pickup_hour
    ),
    ranked_hours AS (
        SELECT
            pickup_hour,
            trip_count,
            ROW_NUMBER() OVER (
                ORDER BY trip_count DESC
            ) AS busiest_rank,
            ROW_NUMBER() OVER (
                ORDER BY trip_count ASC
            ) AS slowest_rank
        FROM hourly_counts
    )

    SELECT
        'Busiest hour' AS result,
        pickup_hour,
        trip_count
    FROM ranked_hours
    WHERE busiest_rank = 1

    UNION ALL

    SELECT
        'Slowest hour' AS result,
        pickup_hour,
        trip_count
    FROM ranked_hours
    WHERE slowest_rank = 1
""")

hours_sql.show(truncate=False)

+------------+-----------+----------+
|result      |pickup_hour|trip_count|
+------------+-----------+----------+
|Busiest hour|18         |515374    |
|Slowest hour|4          |61423     |
+------------+-----------+----------+



In [30]:
# Part 3 - SQL Question 3
# Count pickups by borough using a SQL join.

pickups_by_borough_sql = spark.sql("""
    SELECT
        COALESCE(
            zones.Borough,
            'Unknown'
        ) AS pickup_borough,
        COUNT(*) AS pickup_count
    FROM trips_2019 AS trips
    LEFT JOIN taxi_zones AS zones
        ON trips.PULocationID = zones.LocationID
    GROUP BY
        COALESCE(
            zones.Borough,
            'Unknown'
        )
    ORDER BY
        pickup_count DESC
""")

pickups_by_borough_sql.show(truncate=False)

+--------------+------------+
|pickup_borough|pickup_count|
+--------------+------------+
|Manhattan     |6950511     |
|Queens        |471113      |
|Unknown       |159807      |
|Brooklyn      |91896       |
|Bronx         |18056       |
|N/A           |3890        |
|EWR           |446         |
|Staten Island |361         |
+--------------+------------+



##### Part 3 - Conclusion

I repeated three questions using Spark SQL instead of the PySpark
DataFrame API.

The average passenger count is 1.567. The busiest hour is 6 PM with
515,374 trips, and the slowest hour is 4 AM with 61,423 trips.

For the last question, I used a SQL join between the trip table and
the taxi zone table. Manhattan has the most pickups with 6,950,511
trips.

The SQL results are the same as the results obtained with PySpark.
This shows that both methods can be used to perform the same analysis.

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing